# E-commerce Olist — EDA + Mineração
Objetivo: segmentar clientes (RFM + KMeans k=4), prever satisfação (árvore + matriz de confusão) e extrair associação (co-ocorrência de categorias) para logística e marketing.

> **AVISO — dados sintéticos ilustrativos.** Sem credencial Kaggle neste ambiente, este estudo usa dados **sintéticos realistas** (seed 42) com o esquema da coleção Olist — os resultados abaixo são **ilustrativos**, não do dataset real. Fonte real: `kaggle datasets download olistbr/brazilian-ecommerce`. Com credenciais, baixe os 9 CSVs para `data/` e reexecute este notebook para regenerar tudo com dados reais.

In [1]:
import sys
from pathlib import Path
ROOT = Path('.').resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent  # executado a partir de notebooks/
sys.path.insert(0, str(ROOT))
SAMPLE_CSV = ROOT / 'docs' / 'assets' / 'data' / 'sample_olist.csv'
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.common import df_to_html_table, load_csv, save_fig
from src.mining import confusion_summary, run_kmeans, run_tree

## 1. Entendimento — dicionário dos 9 CSVs
Fonte real: `kaggle datasets download olistbr/brazilian-ecommerce`. Tabela via `df_to_html_table`:

In [2]:
dict9 = pd.DataFrame([
    ("olist_orders_dataset.csv", "pedidos: status, timestamps de compra/aprovacao/entrega"),
    ("olist_order_items_dataset.csv", "itens: preco, frete, vendedor por pedido/produto"),
    ("olist_customers_dataset.csv", "clientes: id, cidade/UF"),
    ("olist_products_dataset.csv", "produtos: categoria, dimensoes, peso"),
    ("olist_order_reviews_dataset.csv", "avaliacoes: review_score 1-5"),
    ("olist_order_payments_dataset.csv", "pagamentos: tipo, parcelas, valor"),
    ("olist_sellers_dataset.csv", "vendedores: cidade/UF"),
    ("olist_geolocation_dataset.csv", "geolocalizacao: CEP/lat-lng por cidade"),
    ("product_category_name_translation.csv", "traducao categoria PT-BR -> EN"),
], columns=["arquivo", "conteudo"])
display(HTML(df_to_html_table(dict9, max_rows=20)))

arquivo,conteudo
olist_orders_dataset.csv,"pedidos: status, timestamps de compra/aprovacao/entrega"
olist_order_items_dataset.csv,"itens: preco, frete, vendedor por pedido/produto"
olist_customers_dataset.csv,"clientes: id, cidade/UF"
olist_products_dataset.csv,"produtos: categoria, dimensoes, peso"
olist_order_reviews_dataset.csv,avaliacoes: review_score 1-5
olist_order_payments_dataset.csv,"pagamentos: tipo, parcelas, valor"
olist_sellers_dataset.csv,vendedores: cidade/UF
olist_geolocation_dataset.csv,geolocalizacao: CEP/lat-lng por cidade
product_category_name_translation.csv,traducao categoria PT-BR -> EN


## 2. Preparação — geração (fallback sintético), join, nulos, outliers
Join `orders ⨝ order_items ⨝ customers`. Nulos de `order_delivered_customer_date` / `order_delivered_carrier_date` (~3% cancelados) viram flag `is_null_delivered` e saem do `delay_days`. Outliers de preço/frete (IQR) com clip no p99.

In [3]:
RNG = np.random.default_rng(42)
STATES = ["SP", "RJ", "MG", "RS", "PR", "BA", "SC", "GO", "ES", "PE", "CE", "PA"]
STATE_P = np.array([0.42, 0.13, 0.11, 0.06, 0.05, 0.04, 0.04, 0.03, 0.03, 0.03, 0.03, 0.03])
STATE_P /= STATE_P.sum()
CAPITAL = {"SP": "sao paulo", "RJ": "rio de janeiro", "MG": "belo horizonte",
           "RS": "porto alegre", "PR": "curitiba", "BA": "salvador", "SC": "florianopolis",
           "GO": "goiania", "ES": "vitoria", "PE": "recife", "CE": "fortaleza", "PA": "belem"}
DIST_KM = {"SP": 60, "RJ": 430, "MG": 580, "ES": 880, "PR": 410, "SC": 700,
           "RS": 1100, "GO": 900, "BA": 1450, "PE": 2100, "CE": 2400, "PA": 2900}
CATEGORIES = ["cama_mesa_banho", "beleza_saude", "esporte_lazer", "moveis_decoracao",
              "informatica_acessorios", "utilidades_domesticas", "relogios_presentes",
              "telefonia", "ferramentas_jardim", "automotivo", "brinquedos",
              "perfumaria", "bebes", "papelaria", "cool_stuff"]
CAT_P = np.array([0.11, 0.10, 0.09, 0.09, 0.08, 0.08, 0.07, 0.07, 0.06, 0.06, 0.05, 0.05, 0.04, 0.03, 0.02])
CAT_P /= CAT_P.sum()
N_ORDERS, N_CUSTOMERS = 6000, 4200
customers = pd.DataFrame({
    "customer_id": [f"c{i:05d}" for i in range(N_CUSTOMERS)],
    "customer_unique_id": [f"u{i:05d}" for i in range(N_CUSTOMERS)],
    "customer_state": RNG.choice(STATES, N_CUSTOMERS, p=STATE_P),
})
customers["customer_city"] = customers["customer_state"].map(CAPITAL)
cust_idx = RNG.integers(0, N_CUSTOMERS, N_ORDERS)
order_customers = customers.iloc[cust_idx].reset_index(drop=True)
purchase = pd.to_datetime("2017-01-01") + pd.to_timedelta(RNG.integers(0, 580, N_ORDERS), unit="D")
ship_days = np.clip(RNG.lognormal(2.2, 0.7, N_ORDERS).round(), 1, 60).astype(int)
est_days = ship_days + RNG.integers(-5, 25, N_ORDERS)
cancelled = RNG.random(N_ORDERS) < 0.03  # ~3% sem entrega
orders = pd.DataFrame({
    "order_id": [f"o{i:05d}" for i in range(N_ORDERS)],
    "customer_id": order_customers["customer_id"].to_numpy(),
    "customer_unique_id": order_customers["customer_unique_id"].to_numpy(),
    "customer_state": order_customers["customer_state"].to_numpy(),
    "customer_city": order_customers["customer_city"].to_numpy(),
    "order_status": np.where(cancelled, "canceled", "delivered"),
    "order_purchase_timestamp": purchase,
    "order_delivered_carrier_date": pd.Series(
        purchase + pd.to_timedelta(np.clip(ship_days - 2, 1, None), unit="D")
    ).mask(pd.Series(cancelled)).to_numpy(),
    "order_delivered_customer_date": pd.Series(
        purchase + pd.to_timedelta(ship_days, unit="D")).mask(pd.Series(cancelled)).to_numpy(),
    "order_estimated_delivery_date": purchase + pd.to_timedelta(np.maximum(est_days, 3), unit="D"),
})
orders["delay_days"] = (pd.to_datetime(orders["order_delivered_customer_date"])
    - pd.to_datetime(orders["order_estimated_delivery_date"])).dt.days
PROBS = {0: [0.02, 0.03, 0.06, 0.22, 0.67],
         1: [0.04, 0.05, 0.10, 0.28, 0.53],
         2: [0.15, 0.12, 0.16, 0.24, 0.33]}
bucket = np.where(orders["delay_days"].isna(), -1,
                  np.where(orders["delay_days"] <= 0, 0,
                           np.where(orders["delay_days"] <= 7, 1, 2)))
orders["review_score"] = [np.nan if b == -1 else int(RNG.choice([1, 2, 3, 4, 5], p=PROBS[b]))
                          for b in bucket]  # nota correlacionada com atraso
print(f"pedidos={len(orders)} entregues={orders['order_status'].eq('delivered').sum()}")

pedidos=6000 entregues=5814


In [4]:
item_rows, prod_seq = [], 0
for i, oid in enumerate(orders["order_id"]):  # 15% com 2-3 itens (p/ co-ocorrencia)
    n = 1 if RNG.random() > 0.15 else int(RNG.integers(2, 4))
    for _ in range(n):
        cat = RNG.choice(CATEGORIES, p=CAT_P)
        price = round(float(np.clip(RNG.lognormal(4.6, 0.9), 8, 3000)), 2)
        dist = DIST_KM[orders.loc[i, "customer_state"]]
        freight = round(float(np.clip(8 + 0.012 * dist + RNG.normal(0, 6), 3, 300)), 2)
        if RNG.random() < 0.01:
            freight = round(freight * 5, 2)  # outlier realista de frete
        item_rows.append((oid, f"p{prod_seq:05d}", cat, price, freight))
        prod_seq += 1
items = pd.DataFrame(item_rows, columns=["order_id", "product_id",
                                        "product_category_name", "price", "freight_value"])
df = orders.merge(items, on="order_id", how="left")
df["is_null_delivered"] = df["order_delivered_customer_date"].isna()
print("nulos delivered_customer:", int(df["is_null_delivered"].sum()))
for col in ["price", "freight_value"]:  # outliers IQR -> clip p99
    q1, q99 = df[col].quantile(0.25), df[col].quantile(0.99)
    n_out = int((df[col] > q1 + 1.5 * (df[col].quantile(0.75) - q1)).sum())
    print(f"{col}: {n_out} outliers IQR, clip p99={q99:.2f}")
    df[col] = df[col].clip(upper=q99)
df["freight_ratio"] = df["freight_value"] / df["price"]
df["is_late"] = (df["delay_days"] > 0).fillna(False).astype(int)
display(HTML(df_to_html_table(df[["order_id", "customer_state", "product_category_name",
    "price", "freight_value", "delay_days", "review_score"]].head(10), max_rows=10)))

nulos delivered_customer: 234
price: 1153 outliers IQR, clip p99=764.00
freight_value: 1006 outliers IQR, clip p99=51.00


order_id,customer_state,product_category_name,price,freight_value,delay_days,review_score
o00000,SP,automotivo,96.69,17.07,-24.0,3.0
o00001,PR,beleza_saude,68.74,7.11,-22.0,5.0
o00002,PA,ferramentas_jardim,182.72,38.81,-11.0,5.0
o00003,RJ,ferramentas_jardim,355.98,15.59,-1.0,4.0
o00004,RJ,moveis_decoracao,144.87,7.70,-23.0,5.0
o00005,SP,utilidades_domesticas,173.77,18.45,-4.0,5.0
o00006,SP,moveis_decoracao,109.16,15.56,-5.0,5.0
o00007,SP,papelaria,149.93,3.00,-21.0,5.0
o00007,SP,relogios_presentes,100.94,8.36,-21.0,5.0
o00007,SP,perfumaria,40.36,18.08,-21.0,5.0


## 3. EDA — atraso × review, receita por categoria/UF, frete × distância (7 figs `olist_*` via `save_fig`)

In [5]:
g = df.dropna(subset=['review_score']).groupby('review_score')['delay_days'].mean()
print(g.round(2).to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(g.index.astype(int).astype(str), g.values, color="#1a5fb4")
ax.set(xlabel="review_score", ylabel="atraso medio (dias)", title="Atraso medio por nota")
fig.tight_layout(); save_fig(fig, 'olist_atraso_review.png'); plt.close(fig)
gcat = df.groupby('product_category_name')['price'].sum().sort_values(ascending=True).tail(10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(gcat.index, gcat.values, color="#2ca02c")
ax.set(xlabel="receita (R$)", title="Receita por categoria (top 10)")
fig.tight_layout(); save_fig(fig, 'olist_receita_categoria.png'); plt.close(fig)

{1.0: -7.42, 2.0: -8.95, 3.0: -7.92, 4.0: -9.35, 5.0: -9.86}


In [6]:
guf = df.groupby('customer_state')['price'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(guf.index, guf.values, color="#ff7f0e")
ax.set(xlabel="UF", ylabel="receita (R$)", title="Receita por UF (top 10)")
fig.tight_layout(); save_fig(fig, 'olist_receita_uf.png'); plt.close(fig)
d = df[['customer_state', 'freight_value']].dropna().sample(1500, random_state=42)
d = d.assign(dist=d['customer_state'].map(DIST_KM))  # distancia aprox. do CD por UF
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(d['dist'], d['freight_value'], s=6, alpha=0.4)
ax.set(xlabel="distancia aprox. do CD (km, por UF)", ylabel="frete (R$)",
       title="Frete vs distancia aproximada")
fig.tight_layout(); save_fig(fig, 'olist_frete_distancia.png'); plt.close(fig)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].boxplot(items["price"], tick_labels=["price (bruto)"])
axes[0].set_title("Preco antes do clip p99")
axes[1].boxplot(items["freight_value"], tick_labels=["freight (bruto)"])
axes[1].set_title("Frete antes do clip p99")
fig.suptitle("Outliers de preco/frete (bruto -> clip no p99)")
fig.tight_layout(); save_fig(fig, 'olist_outliers_preco_frete.png'); plt.close(fig)

## 4. Mineração — KMeans em RFM (k=4), árvore p/ `review_score>=4` + `confusion_summary`, co-ocorrência de categorias

In [7]:
ref = pd.to_datetime(df['order_purchase_timestamp']).max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda s: (ref - pd.to_datetime(s).max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('price', 'sum')).reset_index()
X = StandardScaler().fit_transform(rfm[['recency', 'frequency', 'monetary']])
labels, km = run_kmeans(pd.DataFrame(X), k=4, seed=42)  # KMeans: 4 segmentos
rfm['cluster'] = labels
prof = rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].mean().round(2)
prof['n_clientes'] = rfm.groupby('cluster').size().values
print(prof)
display(HTML(df_to_html_table(prof.reset_index(), max_rows=20)))
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(rfm['recency'], rfm['monetary'], c=rfm['cluster'], s=8, alpha=0.5, cmap='tab10')
ax.set(xlabel="recencia (dias)", ylabel="monetario (R$)",
       title="Segmentos KMeans em RFM (k=4)")
fig.colorbar(sc, label='cluster')
fig.tight_layout(); save_fig(fig, 'olist_clusters_rfm.png'); plt.close(fig)

         recency  frequency  monetary  n_clientes
cluster                                          
0         420.23       1.24    194.68        1006
1         151.58       2.57    476.00         837
2         121.42       3.75    972.69         295
3         132.18       1.39    160.83        1076


cluster,recency,frequency,monetary,n_clientes
0,420.23,1.24,194.68,1006
1,151.58,2.57,476.00,837
2,121.42,3.75,972.69,295
3,132.18,1.39,160.83,1076


In [8]:
feat = df.dropna(subset=['review_score']).copy()
feat['delay_fill'] = feat['delay_days'].fillna(0)
feat['n_items'] = feat.groupby('order_id')['product_id'].transform('count')
Xf = feat[['delay_fill', 'freight_value', 'price', 'freight_ratio', 'n_items', 'is_late']]
yf = (feat['review_score'] >= 4).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=42, stratify=yf)
tree = run_tree(Xtr, ytr)
summ = confusion_summary(yte, tree.predict(Xte))
print(f"accuracy={summ['accuracy']:.4f} confusion_matrix={summ['confusion_matrix']}")
print(pd.Series(tree.feature_importances_, index=Xf.columns).sort_values(ascending=False).round(3).to_dict())
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(summ['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set(xlabel='predito (1=review>=4)', ylabel='real')
ax.set_title(f"Matriz de confusao — arvore (acc={summ['accuracy']:.2f})")
fig.tight_layout(); save_fig(fig, 'olist_matriz_confusao.png'); plt.close(fig)

accuracy=0.7968 confusion_matrix=[[42, 198], [235, 1656]]
{'price': 0.278, 'freight_value': 0.27, 'freight_ratio': 0.265, 'delay_fill': 0.158, 'n_items': 0.029, 'is_late': 0.0}


In [9]:
multi = items.groupby('order_id')['product_category_name'].apply(lambda s: sorted(set(s)))
pairs = [p for cats in multi if len(cats) > 1 for p in itertools.combinations(cats, 2)]
cooc = (pd.DataFrame(pairs, columns=['cat_a', 'cat_b']).value_counts()
        .head(10).reset_index(name='n_pedidos'))
display(HTML(df_to_html_table(cooc, max_rows=10)))

cat_a,cat_b,n_pedidos
beleza_saude,cama_mesa_banho,42
cama_mesa_banho,relogios_presentes,37
beleza_saude,esporte_lazer,36
cama_mesa_banho,utilidades_domesticas,33
cama_mesa_banho,esporte_lazer,32
cama_mesa_banho,informatica_acessorios,31
cama_mesa_banho,moveis_decoracao,29
beleza_saude,utilidades_domesticas,28
esporte_lazer,utilidades_domesticas,27
beleza_saude,telefonia,27


## 5. Amostra publicada — 200 linhas em `docs/assets/data/sample_olist.csv` (round(2), ≤200KB; leitura verificada com `load_csv`)

In [10]:
sample = (df[['order_id', 'customer_state', 'product_category_name', 'price',
              'freight_value', 'delay_days', 'review_score', 'is_late']]
          .sample(200, random_state=42).sort_values('order_id').round(2))
SAMPLE_CSV.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(SAMPLE_CSV, index=False)
check = load_csv(SAMPLE_CSV)
print(check.shape, check.head(3).to_dict('records'))
import os; print('bytes:', os.path.getsize(SAMPLE_CSV))

(200, 8) [{'order_id': 'o00031', 'customer_state': 'GO', 'product_category_name': 'telefonia', 'price': 109.22, 'freight_value': 13.44, 'delay_days': -14.0, 'review_score': 5.0, 'is_late': 0}, {'order_id': 'o00048', 'customer_state': 'BA', 'product_category_name': 'cama_mesa_banho', 'price': 133.7, 'freight_value': 30.88, 'delay_days': 2.0, 'review_score': 5.0, 'is_late': 1}, {'order_id': 'o00061', 'customer_state': 'SP', 'product_category_name': 'utilidades_domesticas', 'price': 217.83, 'freight_value': 3.0, 'delay_days': -9.0, 'review_score': 4.0, 'is_late': 0}]
bytes: 9777


## 6. Decisão (se → então)
1. Se atraso > 7 dias derruba a nota, então reforçar CD no Sudeste (recomenda-se estoque regional nas UFs de maior receita).
2. Se o KMeans isola o segmento de alto valor, então fidelizá-lo com frete progressivo (recomenda-se programa de recompra).
3. Se razão frete/preço alta prediz nota baixa, então subsidiar frete de longa distância (recomenda-se rever a tabela acima de 1500 km).
4. Se categorias co-ocorrem, então criar bundles com desconto (recomenda-se kits das categorias associadas).

Com dados reais, reexecute este notebook após `kaggle datasets download olistbr/brazilian-ecommerce` e valide se os limiares (7 dias, k=4) se mantêm.